# ExoQ Pipeline Tutorial

This notebook demonstrates how to use the ExoQ data pipeline modules independently.

## Overview

The ExoQ pipeline consists of 7 modules:
1. **Data Input** - Load coordinates
2. **Start Exoplanet Quest** - Check NASA Exoplanet Archive
3. **TESS Light Curves** - Retrieve observation data
4. **Transit Detection** - Detect transits with BLS
5. **Habitability Scoring** - Score habitability
6. **Results Summary** - Generate summary
7. **Data Export** - Export results

Each module can run independently with mock data for testing.

## Setup

Import the required modules and add the src directory to the path.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import pandas as pd
import numpy as np

from modules.module1_data_input import DataInputModule
from modules.module3_exoplanet_crossmatch import StartExoplanetQuestModule
from modules.module4_tess_lightcurves import TESSLightCurveModule
from modules.module5_transit_detection import TransitDetectionModule
from modules.module6_habitability_scoring import HabitabilityScoringModule
from modules.module7_results_summary import ResultsSummaryModule
from modules.module8_data_export import DataExportModule

print("✅ Modules imported successfully")

## Module 1: Data Input and Gaia Survival Test

Load coordinates from various sources: virgin list, vetted list, manual entry, or TIC IDs.

In [ ]:
# Initialize Module 1
module1 = DataInputModule()

# Load from virgin list (simulated data)
df, validation = module1.load_from_virgin_list(n_stars=10)

# Display success summary
print(module1.get_success_summary())

# Display sample data
print("\nSample data:")
print(df[['source_id', 'ra', 'dec']].head())

## Module 2: Start Exoplanet Quest

Cross-match stars with NASA Exoplanet Archive to identify known exoplanets.

In [ ]:
# Initialize Module 2
module2 = StartExoplanetQuestModule()

# Cross-match (using mock data)
df, report = module2.cross_match(df, use_mock=True)

# Display success summary
print(module2.get_success_summary())

# Display sample data
print("\nCross-match results:")
print(df[['source_id', 'has_exoplanet', 'exo_pl_name', 'exo_pl_orbper']].head())

## Module 2: Exoplanet Cross-Match

Cross-match with NASA Exoplanet Archive to identify known exoplanets.

In [ ]:
# Initialize Module 2
module2 = StartExoplanetQuestModule()

# Cross-match (using mock data)
df, report = module2.cross_match(df, use_mock=True)

# Display success summary
print(module2.get_success_summary())

# Display sample data
print("\nCross-match results:")
print(df[['source_id', 'has_exoplanet', 'exo_pl_name', 'exo_pl_orbper']].head())

## Module 3: TESS Light Curves

Retrieve TESS light curves from MAST API.

In [ ]:
# Initialize Module 3
module3 = TESSLightCurveModule()

# Retrieve light curves (using mock data)
df, report = module3.retrieve_lightcurves(df, use_mock=True)

# Display success summary
print(module3.get_success_summary())

# Display sample data
print("\nLight curve metadata:")
print(df[['source_id', 'tess_available', 'sectors', 'data_points', 'cadence_minutes']].head())

## Module 4: Transit Detection

Detect transits using BLS periodogram.

In [ ]:
# Initialize Module 4
module4 = TransitDetectionModule()

# Detect transits (using mock data)
df, report = module4.detect_transits(df, use_mock=True)

# Display success summary
print(module4.get_success_summary())

# Display sample data
print("\nTransit detection results:")
print(df[['source_id', 'has_transit_candidate', 'transit_period', 'transit_snr']].head())

## Module 5: Habitability Scoring

Score habitability of stars and exoplanets.

In [ ]:
# Initialize Module 5
module5 = HabitabilityScoringModule()

# Score habitability
df, report = module5.score_habitability(df, df)

# Display success summary
print(module5.get_success_summary())

# Display sample data
print("\nHabitability scores:")
print(df[['source_id', 'stellar_hab_score', 'exo_hab_score', 'esi']].head())

## Module 6: Results Summary

Generate comprehensive results summary.

In [ ]:
# Initialize Module 6
module6 = ResultsSummaryModule()

# Generate summary
df, report = module6.generate_summary(df)

# Display success summary
print(module6.get_success_summary())

# Display top discoveries
print("\nTop Discoveries:")
for i, discovery in enumerate(report['top_discoveries'][:5], 1):
    print(f"{i}. TIC {discovery['source_id']} - {discovery['description']}")

## Module 7: Data Export

Export results in multiple formats.

In [ ]:
# Initialize Module 7
module7 = DataExportModule()

# Export data (CSV and JSON)
report, summary = module7.export_data(df, formats=['csv', 'json'])

# Display success summary
print(summary)

# Display export report
print("\nExport Report:")
print(f"Formats exported: {report['n_formats']}")
print(f"Rows exported: {report['n_rows_exported']}")
print(f"Total size: {report['total_size_kb']:.2f} KB")

## Full Pipeline Example

Run all modules in sequence using convenience functions.

In [ ]:
# Convenience functions for quick usage
from modules.module1_data_input import load_data
from modules.module3_exoplanet_crossmatch import cross_match_exoplanets
from modules.module4_tess_lightcurves import retrieve_tess_lightcurves
from modules.module5_transit_detection import detect_transits
from modules.module6_habitability_scoring import score_habitability
from modules.module7_results_summary import generate_summary
from modules.module8_data_export import export_data

# Run full pipeline
print("="*70)
print("EXOQ FULL PIPELINE")
print("="*70)

# Module 1
df, summary = load_data(input_type='virgin', n_stars=5)
print(summary)

# Module 2
df, summary = cross_match_exoplanets(df, use_mock=True)
print(summary)

# Module 3
df, summary = retrieve_tess_lightcurves(df, use_mock=True)
print(summary)

# Module 4
df, summary = detect_transits(df, use_mock=True)
print(summary)

# Module 5
df, summary = score_habitability(df, df)
print(summary)

# Module 6
df, summary = generate_summary(df)
print(summary)

# Module 7
report, summary = export_data(df, formats=['csv', 'json'])
print(summary)

print("\n" + "="*70)
print("PIPELINE COMPLETE")
print("="*70)

## Using Your Own Data

You can also use your own data with the pipeline.

In [ ]:
# Example: Load from CSV file
# module1 = DataInputModule()
# df, validation = module1.load_csv('path/to/your/data.csv')

# Example: Manual coordinate entry
# coordinates = [
#     {'ra': 150.0, 'dec': 10.0},
#     {'ra': 200.0, 'dec': -20.0}
# ]
# df, validation = module1.load_manual_entry(coordinates)

# Example: Load from TIC IDs
# tic_ids = [123456789, 987654321]
# df, validation = module1.load_tic_ids(tic_ids)

print("See comments above for examples of using your own data.")

## Notes

- All modules support `use_mock=True` for offline testing
- Set `use_mock=False` to query real APIs (requires internet)
- Each module has a `get_success_summary()` method for congratulatory feedback
- Modules can be run independently or in sequence
- Data is passed between modules as pandas DataFrames

## Next Steps

- Try with real data by setting `use_mock=False`
- Adjust parameters (e.g., period range, S/N threshold)
- Visualize results with matplotlib/plotly
- Integrate into your own applications